# Open-AI Project Diffusion Across Global Cities: Roles, Adoption Speed, and Network-Driven Prediction

---

## Preparation

- [Github link](https://github.com/xxx/CASA0006-DSSS-Assessment)

- Number of words: ***

- Runtime: ~0.5 hours (*Memory 16 GB, CPU Apple M-series or Intel i7*)

- Coding environment: Python 3.10+ with Jupyter Notebook

- License: this notebook is made available under the [Creative Commons Attribution license](https://creativecommons.org/licenses/by/4.0/).

- Additional library *[libraries not included in SDS Docker or not used in this module]*:
    - **xgboost**: Gradient boosting framework for event-level adoption-speed modelling (§8).
    - **shap**: SHAP (SHapley Additive exPlanations) for tree-model interpretability (§8).
    - **torch** & **torch_geometric**: PyTorch and PyTorch Geometric for GraphSAGE city-network prediction (§9).
    - **watermark**: A Jupyter Notebook extension for printing timestamps, version numbers, and hardware information.

- Data preparation: All raw data collection (GitHub API, Hugging Face Hub API) and geocoding were conducted offline via 15 standalone Python scripts in the `ELSE/scripts/` directory. This notebook reads only the pre-processed CSV outputs stored in `data/output/` and `data/processed/`. A companion notebook (`Data_Preparation_Pipeline.ipynb`) documents the full pipeline.

---

## Table of contents

1. [Introduction](#Introduction)

1. [Research questions](#Research-questions)

1. [Data](#Data)

1. [Methodology](#Methodology)

1. [Results and discussion](#Results-and-discussion)

1. [Conclusion](#Conclusion)

1. [References](#References)

---

## Introduction

[[ go back to the top ]](#Table-of-contents)

In [ ]:
The rise of open-source artificial intelligence has reshaped the global technology landscape. Platforms such as GitHub and Hugging Face enable developers worldwide to create, share, and build upon AI models, datasets, and applications with minimal barriers to entry. At first glance, this digital infrastructure appears to flatten geography: anyone with an internet connection can, in principle, contribute to or adopt a prominent open-AI project. Yet a growing body of evidence suggests that the **creation, collaboration, and adoption** of open-source technologies remain **highly concentrated** in a small number of global cities, reproducing — and sometimes reinforcing — long-standing spatial inequalities in innovation capacity (Feldman & Audretsch, 1999; Balland et al., 2020).

Several strands of literature inform this investigation. First, research on the **geography of open-source software** has shown that GitHub activity clusters in major tech hubs, with a pronounced core–periphery structure: a handful of cities account for a disproportionate share of repository creation and contributor participation (Wachs et al., 2022). Second, the broader **innovation geography** literature demonstrates that complex economic activities — including those at the frontier of AI — disproportionately concentrate in large, well-connected cities, reflecting the persistent role of agglomeration economies and knowledge spillovers even in the digital age (Balland et al., 2020; Boschma, 2005). Third, the **innovation diffusion** literature, rooted in Rogers' (2003) classic framework, highlights the role of network position and adopter characteristics in determining how quickly new technologies spread across spatial units. More recently, **graph neural network** approaches such as GraphSAGE have been applied to predict node-level outcomes in networked settings (Hamilton et al., 2017), opening new possibilities for urban-scale diffusion modelling.

Despite these advances, most existing studies focus on a **single platform** (e.g., GitHub alone), a **narrow technology family** (e.g., one specific AI model lineage), or **national-level** aggregations that obscure city-level heterogeneity. Few studies have attempted to characterise the full ecosystem of **prominent open-AI projects** — spanning both GitHub repositories and Hugging Face models — at the **city level**, nor have they systematically examined how city characteristics and inter-city collaboration networks jointly shape the speed and breadth of project diffusion across the global urban system.

This project addresses these gaps through an integrated analysis of **23,481 prominent open-AI projects** across **148 global cities** during 2022–2025. It makes four main contributions. First, it identifies **distinct city roles** (originators, collaboration hubs, bridges, and late adopters) using K-means clustering with systematic robustness checks (DBSCAN, GMM, parameter sensitivity). Second, it examines **which city characteristics** — including network centrality, developer base, GDP, education, R&D, and digital infrastructure — are associated with innovation-oriented roles and project adoption breadth via OLS regression with regularisation robustness. Third, it investigates **adoption speed** at the event level using XGBoost with SHAP-based interpretability, revealing that project-side heterogeneity dominates city-side effects. Fourth, it leverages **GraphSAGE** on the city collaboration network to predict future adoption intensity, demonstrating that graph structure provides a significant predictive increment over node features alone. Finally, a technology-type interaction analysis (RQ5) tests whether the effects of city characteristics on diffusion speed vary across AI sub-domains (LLM foundation, LLM application, vision, agent, multimodal, speech).

---

## Research questions

[[ go back to the top ]](#Table-of-contents)

In [ ]:
### Main research question

**How do global cities participate in the creation and adoption of prominent open-AI projects, and what factors shape the diffusion of these projects across the global urban system?**

### Sub-questions

1. **RQ1 — City roles:** What distinct roles do global cities play in the ecosystem of prominent open-AI projects, such as originators, collaboration hubs, bridge cities, and late adopters?

2. **RQ2 — Innovation and adoption breadth:** Which city characteristics are associated with innovation-oriented roles and project adoption breadth within the ecosystem of prominent open-AI projects?

3. **RQ3 — Adoption speed:** Which city characteristics are associated with faster adoption speed of prominent open-AI projects across cities?

4. **RQ4 — Network-based prediction:** Can the city collaboration network derived from GitHub help predict which cities will become the next adopters of prominent open-AI projects?

5. **RQ5 — Technology-type heterogeneity:** Do the effects of city characteristics on the diffusion speed of prominent open-AI projects vary across technology types (e.g., LLM foundation models, LLM applications, vision, agent, multimodal, speech)?

### Mapping of research questions to methods

| RQ | Method | Section |
|:---|:-------|:--------|
| RQ1 | K-means clustering (k=4) + DBSCAN / GMM robustness | §6 |
| RQ2 | OLS regression (Models B/C) + Ridge / Lasso / ElasticNet robustness | §7 |
| RQ3 | XGBoost event-level regression + SHAP interpretability | §8 |
| RQ4 | GraphSAGE (2-layer) + MLP ablation baseline | §9 |
| RQ5 | OLS interaction regression (54 city × tech-type terms) | §10 |

---

## Data

[[ go back to the top ]](#Table-of-contents)

### 3.1 Data Sources

| Dataset | Source | Spatial coverage | Temporal coverage |
|---|---|---|---|
| GitHub repository & activity signals | GitHub REST API (metadata, contributors, commit/PR participation where collected) | Global; restricted to prominent candidates and users geocoded into the 148-city list | Event and `created_at` timestamps; monthly aggregation (YYYYMM) in core tables |
| Hugging Face model signals | Hugging Face Hub metadata & author fields | Global; restricted to prominent candidates | Model `created_at`; monthly aggregation where used |
| Curated target cities | In-repository `city_list.csv` (Step 8 in `Data_Preparation_Pipeline.ipynb`) | 148 cities, 50 countries, 9 macro-regions | Static study frame (versioned with pipeline) |
| Prominent project universe | Step 3 in `Data_Preparation_Pipeline.ipynb` → `prominent_projects_master.csv` | Unified GitHub + HF project identifiers | Per-project creation time (ISO 8601) |
| External city covariates | Step 12 in `Data_Preparation_Pipeline.ipynb` (e.g. WB-class proxies, QS counts) | City- or country-level joins to the 148 cities | ~2022–2023 reference values |
| HF derivation edges (optional; run before Step 11) | Step 10 in `Data_Preparation_Pipeline.ipynb` | Project–project ancestor links | Descendant / ancestor creation months |


### 3.2 Data Preparation

| Category | Source / method | Description & preprocessing |
|---|---|---|
| Prominent project filter | Step 3 in `Data_Preparation_Pipeline.ipynb` — merge of `github_candidates` and `hf_candidates` | Retains `prominent_flag = 1` only; unified schema for platforms, metrics, and AI-relevance fields. |
| Contributor / participation scrape | Step 9 in `Data_Preparation_Pipeline.ipynb` | GitHub contributor lists and event timing used where available for adoption chronology. |
| User–city mapping | Step 4–5 (owner/author locations) plus Steps 6–8 in `Data_Preparation_Pipeline.ipynb` (Step 7 geocoding as needed) | Raw location strings cleaned and matched to `city_list`; only target cities enter adoption and edge construction. |
| Adoption events | Step 11 in `Data_Preparation_Pipeline.ipynb` | For each (city, project), first linking month (GitHub contributors/participation dates when available; else rule-based fallback); global origin month; non-negative lag; `is_originator` from owner vs contributor logic (plus HF derivation supplement when Step 10 edges exist). |
| Collaboration edges | Step 11 in `Data_Preparation_Pipeline.ipynb` | Undirected city pairs: shared contributors on the same prominent repo (GitHub) plus distinct HF derivation pairs; `edge_weight` counts shared “project units”; monthly snapshots attribute edges to project months. |
| City attributes | Steps 11–13 in `Data_Preparation_Pipeline.ipynb` | Sums and network centralities from adoption + aggregated edges; Step 12 merges population, GDP, education, internet, R&D, research capacity, timezone, region, and per-capita rates; Step 13 attaches modelling-oriented enrichments; `cluster` / `role` appended only in later analysis stages. |


### 3.3 Variables Selected

This subsection inventories **column names as delivered in the five pipeline CSV outputs** (`data/processed/prominent_projects_master.csv`, `data/output/city_project_adoption_events.csv`, `data/output/city_attributes.csv`, `data/output/city_collaboration_edges.csv`, `data/output/city_collaboration_edges_monthly.csv`). **Notebook-only constructions** — logarithms, z-scores, train-window aggregates, interaction terms, model-internal tensors, etc. — are **not** listed here because they do not appear as separate CSV columns.

Column counts below sum to **67** across the five tables (15 + 6 + 38 + 4 + 4). Types use *Binary*, *Integer*, *Continuous*, *Categorical*, and *Ordinal*.

**Prominent projects master (15)**

| Variable | Type | Definition |
|---|---|---|
| `project_id` | Categorical | Unified project identifier (e.g. gh_*, hf_*). |
| `platform` | Categorical | GitHub or HuggingFace. |
| `full_id` | Categorical | Native platform string (full repo name or HF model id). |
| `project_name` | Categorical | Short display name. |
| `hf_type` | Categorical | Hugging Face resource type (empty for GitHub). |
| `tags` | Categorical | Tag string (often semicolon-separated). |
| `metric_stars` | Integer | Star count (GitHub; often missing for HF). |
| `metric_forks` | Integer | Fork count (GitHub; often missing for HF). |
| `metric_downloads` | Integer | Download count (HF; often missing for GitHub). |
| `metric_likes` | Integer | Like count (HF; often missing for GitHub). |
| `created_at` | Categorical | Repository/model creation time (ISO 8601 string). |
| `open_ai_related` | Binary | Flag: project passes open-AI relevance screen. |
| `ai_evidence` | Categorical | Keywords or labels supporting AI relevance. |
| `ai_confidence` | Ordinal | Confidence tier of relevance rule (e.g. high, medium, low). |
| `prominent_flag` | Binary | Retained rows have prominent_flag = 1 only. |

**City–project adoption events (6)**

| Variable | Type | Definition |
|---|---|---|
| `city` | Categorical | City name. |
| `project_id` | Categorical | Project key (GitHub owner/repo or Hugging Face hf_* id). |
| `global_origin_month` | Integer | Global project origin month as YYYYMM. |
| `city_first_adoption_month` | Integer | First month the city links to the project (YYYYMM). |
| `lag` | Integer | Months from global_origin_month to city_first_adoption_month (non-negative). |
| `is_originator` | Binary | 1 if the city originated the project; 0 otherwise. |

**City-level attributes (38)** — column order matches `city_attributes.csv`.

| Variable | Type | Definition |
|---|---|---|
| `city` | Categorical | Standardised city name; joins to city_list.matched_city. |
| `country` | Categorical | Country name. |
| `lat` | Continuous | Latitude of city centroid (degrees). |
| `lon` | Continuous | Longitude of city centroid (degrees). |
| `entity_count` | Integer | Count of GitHub/HF entities linked to the city in the study frame (activity denominator from city_list). |
| `origination_count` | Integer | Number of adoption events where the city is project originator (is_originator = 1). |
| `origination_rate` | Continuous | origination_count divided by entity_count. |
| `adoption_count` | Integer | Distinct project_id with at least one adoption event. |
| `adoption_rate` | Continuous | adoption_count divided by entity_count. |
| `avg_lag` | Continuous | Mean adoption lag in months across all city–project events. |
| `median_lag` | Continuous | Median adoption lag in months. |
| `collaboration_count` | Integer | Sum of edge_weight on all incident collaboration edges (undirected). |
| `weighted_degree` | Continuous | Weighted degree on the collaboration graph (equivalent to collaboration_count here). |
| `betweenness` | Continuous | Weighted betweenness centrality. |
| `eigenvector_centrality` | Continuous | Weighted eigenvector centrality (zero-filled if estimation fails). |
| `population_million` | Continuous | Metro population in millions (external lookup, Step 12). |
| `gdp_per_capita` | Continuous | Country GDP per capita in current USD (~2023, Step 12). |
| `education_tertiary_pct` | Continuous | Country tertiary gross enrolment ratio (%). |
| `internet_users_pct` | Continuous | Country internet users as share of population (%). |
| `rd_expenditure_pct` | Continuous | Country R&D expenditure as % of GDP. |
| `research_capacity` | Integer | Count proxy for local research capacity (e.g. QS top-500 universities in city). |
| `timezone_utc` | Continuous | Approximate UTC offset in hours (0.5 h grid from longitude). |
| `region` | Categorical | Macro-region (e.g. East Asia, Europe, North America). |
| `origination_rate_pop` | Continuous | Origination events per million population. |
| `adoption_rate_pop` | Continuous | Adopted distinct projects per million population. |
| `collaboration_rate_pop` | Continuous | Collaboration weight sum per million population. |
| `lag_std` | Continuous | Std. dev. of adoption `lag` across all events for the city (Step 6b). |
| `avg_lag_nonorig` | Continuous | Mean `lag` restricted to non-originator events (`is_originator` = 0; Step 6b). |
| `cross_region_ratio_calc` | Continuous | Share of the city's collaboration edge weight that crosses macro-regions (Step 6b). |
| `pop_top25_share` | Continuous | Mean share of adopted projects falling in the global top-25% popularity tier (Step 6b). |
| `orig_top25_share` | Continuous | Bayesian-smoothed share of originated projects in the top-25% popularity tier (Step 6b). |
| `lag_quality_corr` | Continuous | Within-city correlation between adoption lag and project popularity percentile (Step 6b). |
| `avg_fork_star_ratio` | Continuous | Mean fork/star ratio over GitHub projects the city engages with (Step 6b). |
| `iei_std` | Continuous | City-level dispersion of inter-event intervals (IEI) derived from adoption chronology (same IEI construction as downstream notebooks when exported into this table). |
| `iei_quality_corr` | Continuous | City-level correlation summarising IEI vs project popularity / quality among events (exported with IEI block). |
| `avg_iei` | Continuous | City-level mean IEI in months (non-originator diffusion pacing; exported with IEI block). |
| `cluster` | Ordinal | K-means cluster id when clustering outputs are merged back into `city_attributes.csv`. |
| `role` | Categorical | Interpretable role label paired with `cluster` when merged back into the CSV. |

**City-pair collaboration edges, aggregated (4)**

| Variable | Type | Definition |
|---|---|---|
| `source_city` | Categorical | Lexicographically smaller endpoint of the undirected pair. |
| `target_city` | Categorical | Other endpoint of the undirected pair. |
| `edge_weight` | Integer | Count of shared project units linking the pair (GitHub co-repo; HF one unit per derivation edge). |
| `shared_projects` | Integer | In this build, incremented with edge_weight (numerically identical). |

**City-pair collaboration edges, monthly snapshots (4)**

| Variable | Type | Definition |
|---|---|---|
| `source_city` | Categorical | Same semantics as aggregated edge table. |
| `target_city` | Categorical | Same semantics as aggregated edge table. |
| `month` | Integer | Attribution month YYYYMM. |
| `edge_weight` | Integer | Indicator coded as **1** when at least one collaboration is attributed in that month for the pair (current export uses 0/1). |

### 3.4 Data Quality

| Stage | Records | Notes |
|---|---|---|
| Curated city list (`city_list.csv`) | 148 cities | High-confidence geocoding / matching (50 countries, 9 macro-regions). |
| `prominent_projects_master.csv` | 23,481 projects | Prominent GitHub + HF rows (excluding header). |
| `city_project_adoption_events.csv` | 39,158 events | City–project adoption / origin rows. |
| `city_attributes.csv` | 148 rows × 38 columns | One row per city (schema §3.3); Step 12 `population_million` missing for 5 cities (~3.4%) in published diagnostics. |
| `city_collaboration_edges.csv` | 9,636 edges | Aggregated undirected city pairs. |
| `city_collaboration_edges_monthly.csv` | 98,147 rows | Month × pair snapshots (excluding header). |
| Join keys | — | `project_id` links events to the master table; `city` links events and attributes; edge tables use paired `city` names consistent with `city_list`. |


---

## Methodology

[[ go back to the top ]](#Table-of-contents)

*[Note: a flow chart that describes the methodology is strongly encouraged - see the example below. This flow chart can be made using Microsoft powerpoint or visio or other software]*

Source: see [link](https://linkinghub.elsevier.com/retrieve/pii/S2210670722004437).

![image.png](attachment:image.png)

---

## Results and discussion

[[ go back to the top ]](#Table-of-contents)

---

## Conclusion

[[ go back to the top ]](#Table-of-contents)

---

## References

[[ go back to the top ]](#Table-of-contents)

In [ ]:
1. Balland, P.-A., Jara-Figueroa, C., Petralia, S. G., Steijn, M. P. A., Rigby, D. L., & Hidalgo, C. A. (2020). Complex economic activities concentrate in large cities. *Nature Human Behaviour*, 4(3), 248–254. https://doi.org/10.1038/s41562-019-0803-3

2. Boschma, R. (2005). Proximity and Innovation: A Critical Assessment. *Regional Studies*, 39(1), 61–74. https://doi.org/10.1080/0034340052000320887

3. Chen, T., & Guestrin, C. (2016). XGBoost: A Scalable Tree Boosting System. In *Proceedings of the 22nd ACM SIGKDD International Conference on Knowledge Discovery and Data Mining* (pp. 785–794). https://doi.org/10.1145/2939672.2939785

4. Feldman, M. P., & Audretsch, D. B. (1999). Innovation in cities: Science-based diversity, specialization and localized competition. *European Economic Review*, 43(2), 409–429. https://doi.org/10.1016/S0014-2921(98)00047-6

5. Hamilton, W. L., Ying, R., & Leskovec, J. (2017). Inductive Representation Learning on Large Graphs. In *Advances in Neural Information Processing Systems* (NeurIPS), 30, 1024–1034.

6. Lundberg, S. M., & Lee, S.-I. (2017). A Unified Approach to Interpreting Model Predictions. In *Advances in Neural Information Processing Systems* (NeurIPS), 30, 4765–4774.

7. Rogers, E. M. (2003). *Diffusion of Innovations* (5th ed.). New York: Free Press.

8. Wachs, J., Nitecki, M., Schueller, W., & Wagner, C. (2022). The geography of open source software: Evidence from GitHub. *Technological Forecasting and Social Change*, 176, 121478. https://doi.org/10.1016/j.techfore.2021.121478